In [1]:
# 10_similarity_metric_comparison.py

# ------------------- INSTALL & IMPORTS -------------------
!pip install -q torch transformers sentence-transformers faiss-cpu pandas tqdm scikit-learn rouge-score nltk bert-score sacrebleu

import pandas as pd, numpy as np, torch, faiss, time, nltk, warnings, logging
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
from tqdm.auto import tqdm
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
from bert_score import score as bert_score
from sacrebleu.metrics import BLEU

# Reduce HuggingFace model init warnings
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

# NLTK downloads
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

warnings.filterwarnings('ignore')

# ------------------- DATA -------------------
df = pd.read_csv('/kaggle/input/mlops-amazon/amazon.csv')

documents = [
    f"""Product: {r['product_name']}
Price: {r['discounted_price']} | Rating: {r['rating']} ({r['rating_count']} reviews)
Description: {r['about_product']}""" 
    for _, r in df.iterrows()
]

TEST_QUERIES = [
{
"query": "Recommend a good fast charging USB-C cable under 300 rupees",
"reference": "The boAt A400 at ₹299 is recommended, supporting 24W-25W fast charging and having a slightly higher customer rating. For a tighter budget, the pTron Solero TB301 at ₹149 supports 15W fast charging."
},
{
"query": "Which cable has the highest rating and supports 60W charging?",
"reference": "The Belkin USB-C to USB-C Fast Charging Cable (60W PD) is the best match, offering a strong 4.5-star rating along with full 60W fast-charging support."
},
{
"query": "What is the best iPhone lightning cable in the list?",
"reference": "For super-fast charging with a USB-C adapter, the Belkin Lightning to USB-C is best. For a reliable and durable standard cable with a good warranty, the Duracell USB-A to Lightning is a great choice. The Hi-Mobiler is the most budget-friendly option."
},
{
"query": "Suggest me some good long lasting headphones",
"reference": "The boAt Bassheads 100 in-ear wired earphones are recommended for longevity, featuring a premium coated wire for sturdiness and a 1-year warranty. They have a 4.1-star rating from over 360,000 reviews and cost between ₹349-₹379."
}
]

# ------------------- METRICS CLASS -------------------
class Metrics:
    def __init__(self):
        self.rouge = rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=True)
        self.bleu = BLEU(effective_order=True)
        self.embedder = SentenceTransformer('all-MiniLM-L6-v2')

    def all(self, pred, ref, ctx):
        r = self.rouge.score(ref, pred)
        metrics = {
            'rouge_1_f1': r['rouge1'].fmeasure,
            'rouge_l_f1': r['rougeL'].fmeasure,
            'bleu': self.bleu.sentence_score(pred, [ref]).score / 100,
            'meteor': meteor_score([word_tokenize(ref.lower())], word_tokenize(pred.lower())),
        }
        P, R, F = bert_score([pred], [ref], model_type="microsoft/deberta-large-mnli", verbose=False)
        metrics['bert_f1'] = F.mean().item()

        e1 = self.embedder.encode(pred)
        e2 = self.embedder.encode(ref)
        metrics['emb_sim'] = util.cos_sim(e1, e2).item()

        c_emb = self.embedder.encode(ctx)
        metrics['faith'] = min(1.0, 0.7 * util.cos_sim(e1, c_emb).item() + 0.3)

        return metrics

    def composite(self, m):
        w = {
            'rouge_1_f1': 0.1,
            'rouge_l_f1': 0.1,
            'bleu': 0.1,
            'meteor': 0.15,
            'bert_f1': 0.25,
            'emb_sim': 0.2,
            'faith': 0.1
        }
        return sum(m[k] * w[k] for k in w)

metrics_calc = Metrics()

# ------------------- RAG CLASS (METRIC-SENSITIVE) -------------------
class RAG:
    def __init__(self, emb_name, generator, metric='cosine'):
        self.emb_name = emb_name
        self.generator = generator
        self.metric = metric

        print(f"Loading embedding model: {emb_name}")
        self.embedder = SentenceTransformer(emb_name)
        dim = self.embedder.encode(["test"]).shape[1]

        if metric == 'cosine':
            self.index = faiss.IndexFlatIP(dim)
            normalize = True
        elif metric == 'euclidean':
            self.index = faiss.IndexFlatL2(dim)
            normalize = False
        else:
            raise ValueError("metric must be 'cosine' or 'euclidean'")

        print(f"Embedding {len(documents)} documents with {metric} metric...")
        batches = [documents[i:i+32] for i in range(0, len(documents), 32)]
        for b in tqdm(batches, desc="Indexing"):
            embs = self.embedder.encode(b, normalize_embeddings=normalize)
            self.index.add(embs)

    def retrieve(self, q, k):
        qe = self.embedder.encode([q], normalize_embeddings=(self.metric == 'cosine'))
        D, I = self.index.search(qe, k)
        ctx = "\n\n".join([documents[i] for i in I[0]])
        return ctx

    def generate(self, q, ctx):
        prompt = f"Context:\n{ctx}\n\nQuestion: {q}\nAnswer:"
        out = self.generator(
            prompt,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.95,
            top_k=50,
            do_sample=True
        )[0]['generated_text']
        ans = out.split("Answer:")[-1].strip()
        return ans

# ------------------- LOAD GENERATOR -------------------
GEN_MODEL = "Qwen/Qwen2.5-7B-Instruct"
print("\nLoading generator...")
generator = pipeline(
    "text-generation",
    model=GEN_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# ------------------- EXPERIMENT -------------------
results = []
METRICS = ['cosine', 'euclidean']
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"

for metric in METRICS:
    print(f"\n{'='*80}\nTESTING SIMILARITY METRIC: {metric}\n{'='*80}")
    rag = RAG(EMBEDDING_MODEL, generator, metric=metric)

    for qd in TEST_QUERIES:
        ctx = rag.retrieve(qd["query"], k=5)
        ans = rag.generate(qd["query"], ctx)
        m = metrics_calc.all(ans, qd["reference"], ctx)
        m['composite'] = metrics_calc.composite(m)
        results.append({**m, "similarity_metric": metric, "query": qd["query"][:60]})

        print("\n------------------------------------------------------------")
        print(f"Similarity Metric: {metric}")
        print(f"Query: {qd['query']}")
        print("\nGenerated Answer:")
        print(ans)
        print(f"\nComposite Score: {m['composite']:.4f}")
        print("------------------------------------------------------------\n")

df_out = pd.DataFrame(results)

# Average composite score per similarity metric
summary = df_out.groupby("similarity_metric")["composite"].mean().sort_values(ascending=False)
print("\n================ FINAL SUMMARY ================\n")
print("Average Composite Scores by Similarity Metric:")
print(summary)

best_metric = summary.idxmax()
print(f"\n🏆 Best Similarity Metric: {best_metric}")

df_out.to_csv("10_similarity_metric_comparison.csv", index=False)
print("\nSimilarity metric comparison saved → 10_similarity_metric_comparison.csv")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 107.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 81.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 3.1 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 14.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 9.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 101.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━

2025-12-05 09:53:25.297710: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764928405.501630      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764928405.559884      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...


------------------------------------------------------------
Similarity Metric: euclidean
Query: Which cable has the highest rating and supports 60W charging?

Generated Answer:
The Ambrane 60W / 3A Type C Fast Charging Unbreakable 1.5m L Shaped Braided Cable, PD Technology, 480Mbps Data Transfer for Smartphones, Tablet, Laptops & other type c devices (ABLC10, Black) has the highest rating at 4.0 and supports 60W charging.

However, it's worth noting that the MI Xiaomi USB Type C HYperCharge Cable 6A 100cm Sturdy and Durable Black supports 120W hypercharging, which is higher than the 60W supported by the Ambrane cable. But based on the question criteria of having the highest rating and supporting 60W charging, the Ambrane cable is the correct answer. 

If you need a cable that supports 120W charging, the MI Xiaomi cable would be the better choice despite its slightly lower rating. 

Would you like to know more about either product?

Composite Score: 0.4359
----------------------------

The following layers were not sharded: encoder.layer.*.attention.output.LayerNorm.bias, encoder.layer.*.attention.output.dense.weight, embeddings.LayerNorm.bias, encoder.layer.*.intermediate.dense.bias, embeddings.word_embeddings.weight, encoder.layer.*.attention.output.LayerNorm.weight, encoder.layer.*.output.dense.bias, encoder.layer.*.attention.self.q_bias, encoder.layer.*.attention.self.pos_q_proj.weight, encoder.layer.*.intermediate.dense.weight, encoder.layer.*.output.LayerNorm.weight, embeddings.LayerNorm.weight, encoder.layer.*.attention.self.v_bias, encoder.rel_embeddings.weight, encoder.layer.*.output.dense.weight, encoder.layer.*.attention.output.dense.bias, encoder.layer.*.output.LayerNorm.bias, encoder.layer.*.attention.self.pos_proj.weight, encoder.layer.*.attention.self.pos_q_proj.bias, encoder.layer.*.attention.self.in_proj.weight



------------------------------------------------------------
Similarity Metric: euclidean
Query: What is the best iPhone lightning cable in the list?

Generated Answer:
Determining the "best" iPhone lightning cable depends on several factors such as price, performance, durability, compatibility, and user reviews. Here's a brief analysis of each product:

1. **Hi-Mobiler iPhone Charger Lightning Cable (₹254)**
   - **Pros**: 
     - 2 Pack for better value.
     - High-quality materials and overcharge protection.
     - Compatible with a wide range of devices including iPhones, iPads, and iPods.
     - Comes with a 2-year warranty.
   - **Cons**:
     - Lower price compared to other options but still provides good features.
     - The 2-pack might be less durable individually.

2. **Belkin Apple Certified Lightning To Type C Cable (₹1,499)**
   - **Pros**:
     - Supports USB Power Delivery for fast charging.
     - Tested for longevity with 10,000+ bends.
     - Compatible with newer 

The following layers were not sharded: encoder.layer.*.attention.output.LayerNorm.bias, encoder.layer.*.attention.output.dense.weight, embeddings.LayerNorm.bias, encoder.layer.*.intermediate.dense.bias, embeddings.word_embeddings.weight, encoder.layer.*.attention.output.LayerNorm.weight, encoder.layer.*.output.dense.bias, encoder.layer.*.attention.self.q_bias, encoder.layer.*.attention.self.pos_q_proj.weight, encoder.layer.*.intermediate.dense.weight, encoder.layer.*.output.LayerNorm.weight, embeddings.LayerNorm.weight, encoder.layer.*.attention.self.v_bias, encoder.rel_embeddings.weight, encoder.layer.*.output.dense.weight, encoder.layer.*.attention.output.dense.bias, encoder.layer.*.output.LayerNorm.bias, encoder.layer.*.attention.self.pos_proj.weight, encoder.layer.*.attention.self.pos_q_proj.bias, encoder.layer.*.attention.self.in_proj.weight



------------------------------------------------------------
Similarity Metric: euclidean
Query: Suggest me some good long lasting headphones

Generated Answer:
Based on your preference for long-lasting headphones, I would recommend the following options:

1. **boAt Rockerz 450 Bluetooth On Ear Headphones with Mic**: 
   - **Price**: ₹1,220
   - **Battery Life**: Upto 15 hours
   - **Features**: 40mm dynamic drivers, ergonomic design, comfortable padded earcushions, integrated controls, dual connection modes (Bluetooth and AUX), 1 year warranty.
   - **Pros**: Long battery life, comfortable wear, good build quality.
   
2. **Noise Buds VS402 Truly Wireless in Ear Earbuds**:
   - **Price**: ₹1,799
   - **Battery Life**: Up to 35 hours (with charging case)
   - **Features**: 10mm driver speaker, environmental noise cancellation (ENC), hyper sync technology, low latency, breathing LED lights, Bluetooth v5.3, 1 year warranty.
   - **Pros**: Extended battery life, excellent sound quality, 